# profile_decodification

Decode the LDA top-10 day-type profiles into VQ-VAE behavioral features and plot the
feature distribution per pattern.


In [ ]:
import os
import pickle
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def find_project_root() -> Path:
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    if cwd.name == "vq-vae_lda_pipeline":
        return cwd.parent.parent
    for p in (cwd, *cwd.parents):
        if (p / "models").is_dir() and (p / "data").is_dir():
            return p
    return cwd


def rel_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


PROJECT_ROOT = find_project_root()

TOP10_CSV = PROJECT_ROOT / "results/lda/tables/lda_topics_top10.csv"

# Decoded VQ-VAE embedding vectors (profile id -> behavioral feature vector)
DECODED_PKL = PROJECT_ROOT / "data/output_vq_vae/decoded_embedding_vectors_a0.pkl"

OUTPUT_FIG = (
    PROJECT_ROOT / "results/lda/figures/decoded_topic_plots/decoded_top10_lda_patterns.png"
)

# Order of features in each decoded vector
FEATURE_NAMES = [
    "sleep_start",
    "location_distance",
    "location_time_home",
    "sleep_duration",
    "activity_walking",
    "app_usage_total",
    "location_clusters_count",
    "steps_steps_total",
    "weekend",
    "practiced_sport",
]


## 1 · Top-10 profiles → decoded features

In [ ]:
terms_topic = pd.read_csv(TOP10_CSV)

with open(DECODED_PKL, "rb") as f:
    embedding_counts = pickle.load(f)

# Map each profile id to its decoded vector and expand into named feature columns
df_final = terms_topic.copy()
df_final["embedding_vector"] = df_final["profile"].map(embedding_counts)
df_final = df_final.dropna(subset=["embedding_vector"]).reset_index(drop=True)

features = pd.DataFrame(df_final["embedding_vector"].tolist(), columns=FEATURE_NAMES)
df_final = pd.concat([df_final.drop(columns=["embedding_vector"]), features], axis=1)
df_final = df_final.drop(columns=["weekend"])

feature_cols = [c for c in FEATURE_NAMES if c != "weekend"]
df_final.head()

## 2 · Feature distribution by day type per pattern

In [ ]:
palette_map = {0: "Greens_d", 3: "Blues_d"}
default_palette = "tab10"

topics = sorted(df_final["topic"].unique())
n_topics = len(topics)
n_cols = 3
n_rows = (n_topics + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 16), sharey=True)
axes = axes.flatten()

for i, topic_id in enumerate(topics):
    ax = axes[i]
    df_topic = df_final[df_final["topic"] == topic_id].copy()
    df_topic["profile"] = df_topic["profile"].astype(str)

    df_melted = df_topic.melt(
        id_vars=["profile"], value_vars=feature_cols,
        var_name="Feature", value_name="Value",
    )

    current_palette = palette_map.get(topic_id, default_palette)
    sns.barplot(data=df_melted, x="Feature", y="Value", hue="profile",
                palette=current_palette, ax=ax)

    ax.set_title(f"Pattern {topic_id}", fontsize=18, fontweight="bold")
    ax.set_ylabel("Value" if i % n_cols == 0 else "")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=10)
    ax.axhline(0, color="black", linewidth=1.2)
    ax.legend(title="Day-Type ID", loc="upper right", fontsize="8",
              ncol=2, frameon=True, edgecolor="gray")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Feature Distribution by Day Type per Pattern (raw)", fontsize=26, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()  # intermediate figure (not saved)

## 3 · Normalize features (z-score)

In [ ]:
from sklearn.preprocessing import StandardScaler

feature_columns = [c for c in df_final.columns if c not in ["topic", "profile"]]

df_final_scal = df_final.copy()
scaler = StandardScaler()
df_final_scal[feature_columns] = scaler.fit_transform(df_final_scal[feature_columns])

df_final_scal.head()

## 4 · Final figure: feature distribution by day type per pattern (normalized)

In [ ]:
topics = sorted(df_final_scal["topic"].unique())
n_topics = len(topics)
n_cols = 3
n_rows = (n_topics + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 16), sharey=True)
axes = axes.flatten()

for i, topic_id in enumerate(topics):
    ax = axes[i]
    df_topic = df_final_scal[df_final_scal["topic"] == topic_id].copy()
    df_topic["profile"] = df_topic["profile"].astype(str)

    df_melted = df_topic.melt(
        id_vars=["profile"], value_vars=feature_cols,
        var_name="Feature", value_name="Value",
    )

    current_palette = palette_map.get(topic_id, default_palette)
    sns.barplot(data=df_melted, x="Feature", y="Value", hue="profile",
                palette=current_palette, ax=ax)

    ax.set_title(f"Pattern {topic_id}", fontsize=18, fontweight="bold")
    ax.set_ylabel("Z-score (Std Dev)" if i % n_cols == 0 else "")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=10)
    ax.axhline(0, color="black", linewidth=1.2)
    ax.legend(title="Day-Type ID", loc="upper right", fontsize="8",
              ncol=2, frameon=True, edgecolor="gray")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Feature Distribution by Day Type per Pattern", fontsize=26, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.95])

OUTPUT_FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUT_FIG, dpi=300, bbox_inches="tight")
print(f"Saved: {rel_path(OUTPUT_FIG)}")
plt.show()